# M8_8.22–M8_8.24 · NumPy, pandas y Matplotlib

Este cuaderno utiliza los archivos compartidos del repositorio del curso:

- `data/input/pozos.csv`
- `data/input/mediciones.csv`
- `data/input/datos_hidrogeologicos.xlsx`
- `data/database/hidrogeologia.sqlite`

## Índice

### M8_8.22 · NumPy
- Listas y arrays
- Dimensiones, `shape`, `size` y `dtype`
- Indexación y slicing
- Valores booleanos y máscaras
- `NaN` y operaciones que ignoran ausentes
- Vectorización
- Arrays de dos dimensiones y ejes

### M8_8.23 · pandas
- Series y DataFrames
- Importación desde CSV y Excel
- Inspección inicial
- Selección con columnas, `loc` e `iloc`
- Filtrado y ordenación
- Creación y modificación de columnas
- Fechas y series temporales
- Datos ausentes
- Agrupación y agregación
- Combinación de DataFrames
- Formato largo y ancho
- Lectura desde SQLite

### M8_8.24 · Matplotlib
- Figura y ejes
- Series temporales
- Scatter plots
- Histogramas
- Comparaciones entre grupos
- Elementos de una figura científica
- Decisiones que pueden inducir a error
- Exportación de figuras

## Preparación

En Google Colab debe clonarse primero el repositorio del curso. Este cuaderno no genera un conjunto alternativo: todo el alumnado trabaja con los mismos archivos.

In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def encontrar_raiz_repositorio():
    carpeta_actual = Path.cwd().resolve()

    for carpeta in [carpeta_actual, *carpeta_actual.parents]:
        if (carpeta / "data" / "input").exists():
            return carpeta

    raise FileNotFoundError(
        "No se encuentra data/input. En Colab, clona primero el repositorio del curso."
    )


ROOT = encontrar_raiz_repositorio()
INPUT = ROOT / "data" / "input"
DATABASE = ROOT / "data" / "database"
OUTPUT = ROOT / "data" / "output"

(OUTPUT / "tables").mkdir(parents=True, exist_ok=True)
(OUTPUT / "figures").mkdir(parents=True, exist_ok=True)

print("Raíz del repositorio:", ROOT)

## Cargar los datos compartidos

`pozos.csv` contiene propiedades relativamente estables de cada punto. `mediciones.csv` contiene observaciones repetidas en el tiempo. La columna `fecha` se convierte explícitamente a `datetime`.

In [ ]:
ruta_pozos = INPUT / "pozos.csv"
ruta_mediciones = INPUT / "mediciones.csv"

pozos = pd.read_csv(ruta_pozos)
mediciones = pd.read_csv(
    ruta_mediciones,
    parse_dates=["fecha"]
)

print("Dimensiones de pozos:", pozos.shape)
print("Dimensiones de mediciones:", mediciones.shape)

display(pozos.head())
display(mediciones.head())

---
# M8_8.22 · Computación científica con NumPy

## 1. Lista de Python y array de NumPy

Una **lista** puede contener objetos de tipos diferentes. Un **array** está diseñado para cálculo numérico y normalmente contiene valores de un mismo tipo.

Un array permite aplicar una operación a todos sus elementos sin escribir un bucle explícito. Esta forma de cálculo se llama **vectorización**.

In [ ]:
niveles_lista = [8.7, 8.4, 9.1, 8.9]
niveles_array = np.array([8.7, 8.4, 9.1, 8.9])

print("Lista:", niveles_lista)
print("Array:", niveles_array)
print("Tipo de la lista:", type(niveles_lista))
print("Tipo del array:", type(niveles_array))

## 2. Dimensión, forma, tamaño y tipo

- **`ndim`**: número de dimensiones.
- **`shape`**: longitud del array en cada dimensión.
- **`size`**: número total de elementos.
- **`dtype`**: tipo numérico almacenado.

En datos hidrogeológicos, un vector puede representar una serie temporal. Una matriz puede representar varios pozos por varias fechas, o una capa de un modelo numérico.

In [ ]:
print("Número de dimensiones:", niveles_array.ndim)
print("Forma:", niveles_array.shape)
print("Número de elementos:", niveles_array.size)
print("Tipo de dato:", niveles_array.dtype)

## 3. Indexación y slicing

NumPy comienza a contar en cero. Un índice selecciona un elemento; un *slice* selecciona un intervalo.

En `inicio:fin`, el elemento de la posición `fin` no se incluye.

In [ ]:
print("Primer elemento:", niveles_array[0])
print("Último elemento:", niveles_array[-1])
print("Elementos de las posiciones 1 y 2:", niveles_array[1:3])
print("Primeros tres elementos:", niveles_array[:3])

## 4. Comparaciones y máscaras booleanas

Una comparación devuelve un array de valores `True` y `False`. Ese array puede utilizarse como máscara para seleccionar observaciones.

In [ ]:
mascara_profundos = niveles_array > 8.8

print("Máscara:", mascara_profundos)
print("Valores seleccionados:", niveles_array[mascara_profundos])

## 5. Valores ausentes con `NaN`

`NaN` representa un valor numérico ausente. Una operación ordinaria puede devolver `NaN`; las funciones `nan*` ignoran valores ausentes.

Ignorar un ausente permite calcular, pero no resuelve la causa de la ausencia. La decisión científica se estudia en M8_8.27.

In [ ]:
niveles_con_ausente = np.array([8.7, 8.4, np.nan, 9.1])

print("Media ordinaria:", np.mean(niveles_con_ausente))
print("Media ignorando NaN:", np.nanmean(niveles_con_ausente))
print("Número de ausentes:", np.isnan(niveles_con_ausente).sum())

## 6. Vectorización

Si la profundidad del nivel se mide desde la superficie, la cota piezométrica puede calcularse restando cada profundidad a la cota del terreno. NumPy aplica la resta elemento por elemento.

In [ ]:
cota_terreno_m = 112.4
profundidades_m = np.array([8.7, 8.4, 9.1, 8.9])

cotas_piezometricas_m = cota_terreno_m - profundidades_m

print("Profundidades:", profundidades_m)
print("Cotas piezométricas:", cotas_piezometricas_m)

## 7. Arrays bidimensionales y ejes

En la matriz siguiente:

- cada fila representa un pozo;
- cada columna representa una fecha.

`axis=0` recorre las filas y devuelve un resultado por columna. `axis=1` recorre las columnas y devuelve un resultado por fila.

In [ ]:
matriz_niveles = np.array([
    [8.1, 8.4, 8.6],
    [14.2, 14.0, 13.8]
])

print("Matriz:")
print(matriz_niveles)
print("Forma:", matriz_niveles.shape)

media_por_fecha = matriz_niveles.mean(axis=0)
media_por_pozo = matriz_niveles.mean(axis=1)

print("Media por fecha:", media_por_fecha)
print("Media por pozo:", media_por_pozo)

### Actividad M8_8.22

1. Crea un array con cinco profundidades, incluyendo un `NaN`.
2. Inspecciona `ndim`, `shape`, `size` y `dtype`.
3. Selecciona las profundidades superiores a un umbral.
4. Calcula cotas piezométricas mediante vectorización.
5. Explica qué representa `axis=0` en una matriz definida por pozos y fechas.

---
# M8_8.23 · Manipulación de datos con pandas

## 8. Series y DataFrames

Una **Series** es una secuencia unidimensional con índice. Un **DataFrame** es una estructura bidimensional con filas, columnas e índice.

Una columna de un DataFrame es una Series. Un DataFrame no es lo mismo que una tabla SQL: puede existir únicamente en memoria y no impone por sí mismo claves o integridad referencial.

In [ ]:
serie_niveles = mediciones["profundidad_nivel_m"]

print("Tipo de una columna:", type(serie_niveles))
print("Tipo del conjunto completo:", type(mediciones))

display(serie_niveles.head())

## 9. Importar CSV

`read_csv` lee un archivo delimitado. Conviene declarar fechas y comprobar siempre dimensiones, nombres de columnas, tipos y primeras filas.

In [ ]:
mediciones_csv = pd.read_csv(
    INPUT / "mediciones.csv",
    parse_dates=["fecha"]
)

print("Dimensiones:", mediciones_csv.shape)
print("Columnas:", mediciones_csv.columns.to_list())
print()
print("Tipos:")
print(mediciones_csv.dtypes)

display(mediciones_csv.head())

## 10. Importar Excel

Un libro Excel puede contener varias hojas. `sheet_name` selecciona la hoja. La lectura debe comprobar nombres de hojas, encabezados, fechas y tipos inferidos.

In [ ]:
ruta_excel = INPUT / "datos_hidrogeologicos.xlsx"

hojas = pd.ExcelFile(ruta_excel).sheet_names
print("Hojas disponibles:", hojas)

pozos_excel = pd.read_excel(
    ruta_excel,
    sheet_name="pozos"
)

mediciones_excel = pd.read_excel(
    ruta_excel,
    sheet_name="mediciones",
    parse_dates=["fecha"]
)

print("Dimensiones de la hoja pozos:", pozos_excel.shape)
print("Dimensiones de la hoja mediciones:", mediciones_excel.shape)

## 11. Inspección inicial

- `head()` muestra las primeras filas.
- `shape` devuelve filas y columnas.
- `columns` devuelve nombres.
- `dtypes` muestra tipos.
- `info()` resume estructura y valores no nulos.
- `describe()` calcula un resumen descriptivo.

Ninguna función sustituye la lectura de metadatos y unidades.

In [ ]:
display(mediciones_csv.head())
print("Forma:", mediciones_csv.shape)
print()
mediciones_csv.info()
print()
display(mediciones_csv.describe())

## 12. Seleccionar columnas y filas

- Corchetes seleccionan una o varias columnas.
- `loc` selecciona mediante etiquetas y condiciones.
- `iloc` selecciona mediante posiciones enteras.

Para principiantes, conviene separar la condición, las columnas y el resultado.

In [ ]:
columnas_nivel = mediciones_csv[[
    "id_pozo",
    "fecha",
    "profundidad_nivel_m"
]]

display(columnas_nivel.head())

primeras_tres_filas = mediciones_csv.iloc[0:3]
display(primeras_tres_filas)

## 13. Filtrado paso a paso

La pregunta es: ¿qué niveles se midieron en P01 durante junio, julio y agosto?

In [ ]:
es_pozo_p01 = mediciones_csv["id_pozo"] == "P01"
es_mes_verano = mediciones_csv["fecha"].dt.month.isin([6, 7, 8])

cumple_ambas_condiciones = es_pozo_p01 & es_mes_verano

columnas_resultado = [
    "id_pozo",
    "fecha",
    "profundidad_nivel_m"
]

p01_verano = mediciones_csv.loc[
    cumple_ambas_condiciones,
    columnas_resultado
]

display(p01_verano)

## 14. Ordenar datos

`sort_values` ordena por una o varias columnas. La ordenación no cambia los valores; cambia la presentación de las filas.

In [ ]:
conductividades_altas = mediciones_csv.sort_values(
    by="conductividad_uScm",
    ascending=False
)

display(conductividades_altas.head())

## 15. Crear una columna

Un cálculo vectorizado puede añadir una columna. Para calcular cota piezométrica necesitamos primero combinar las mediciones con la cota de terreno de cada pozo.

## 16. Combinar DataFrames

`merge` combina DataFrames mediante una clave. `how="left"` conserva todas las mediciones. `validate="many_to_one"` comprueba que muchas mediciones se relacionan con un único pozo.

In [ ]:
datos_completos = mediciones_csv.merge(
    pozos,
    on="id_pozo",
    how="left",
    validate="many_to_one"
)

datos_completos["cota_piezometrica_m"] = (
    datos_completos["cota_terreno_m"]
    - datos_completos["profundidad_nivel_m"]
)

columnas_mostrar = [
    "id_pozo",
    "fecha",
    "cota_terreno_m",
    "profundidad_nivel_m",
    "cota_piezometrica_m"
]

display(datos_completos[columnas_mostrar].head())

## 17. Fechas y series temporales

Una fecha debe tener tipo `datetime`. El accesor `.dt` permite extraer año, mes o día. `set_index` puede convertir la fecha en índice temporal. `resample` cambia la frecuencia y requiere decidir cómo agregar.

In [ ]:
datos_completos["año"] = datos_completos["fecha"].dt.year
datos_completos["mes"] = datos_completos["fecha"].dt.month

serie_p01 = (
    datos_completos.loc[
        datos_completos["id_pozo"] == "P01",
        ["fecha", "profundidad_nivel_m"]
    ]
    .set_index("fecha")
    .sort_index()
)

media_trimestral_p01 = serie_p01.resample("QS").mean()

display(serie_p01.head())
display(media_trimestral_p01)

## 18. Datos ausentes

- `isna()` identifica ausencias.
- `notna()` identifica valores presentes.
- `dropna()` elimina filas según un criterio.
- `fillna()` sustituye ausencias.

Eliminar o rellenar no debe hacerse automáticamente. Aquí solo identificamos ausencias; su tratamiento científico se aborda en M8_8.27.

In [ ]:
ausentes_por_columna = datos_completos.isna().sum()
print("Ausentes por columna:")
print(ausentes_por_columna)

filas_con_nivel_ausente = datos_completos.loc[
    datos_completos["profundidad_nivel_m"].isna()
]

display(filas_con_nivel_ausente)

## 19. Agrupar y agregar

`groupby` divide las filas en grupos y después aplica operaciones. La sintaxis se lee como: agrupar por acuífero, seleccionar profundidad y calcular varios resúmenes.

In [ ]:
grupo_acuifero = datos_completos.groupby("acuifero")
serie_por_grupo = grupo_acuifero["profundidad_nivel_m"]

resumen_acuifero = serie_por_grupo.agg(
    numero_validos="count",
    media="mean",
    mediana="median",
    minimo="min",
    maximo="max"
)

display(resumen_acuifero.round(2))

## 20. Formato largo y ancho

En **formato largo**, cada fila es una observación y una columna identifica el grupo. Es adecuado para filtrar, agrupar y visualizar.

En **formato ancho**, cada pozo ocupa una columna. Puede facilitar la comparación de series, pero resulta menos flexible como estructura principal.

In [ ]:
niveles_largos = datos_completos[[
    "fecha",
    "id_pozo",
    "profundidad_nivel_m"
]]

niveles_anchos = niveles_largos.pivot_table(
    index="fecha",
    columns="id_pozo",
    values="profundidad_nivel_m",
    aggfunc="first"
)

print("Formato largo")
display(niveles_largos.head())

print("Formato ancho")
display(niveles_anchos.head())

## 21. Leer desde SQLite

`read_sql_query` ejecuta una consulta y devuelve el resultado como DataFrame. La tabla permanece en la base; el DataFrame es la copia en memoria utilizada por pandas.

In [ ]:
ruta_sqlite = DATABASE / "hidrogeologia.sqlite"

with sqlite3.connect(ruta_sqlite) as conexion:
    consulta = """
    SELECT
        m.id_pozo,
        m.fecha,
        p.acuifero,
        m.profundidad_nivel_m,
        m.precipitacion_mm,
        m.conductividad_uScm
    FROM mediciones AS m
    INNER JOIN pozos AS p
        ON m.id_pozo = p.id_pozo
    ORDER BY m.id_pozo, m.fecha;
    """

    datos_desde_sqlite = pd.read_sql_query(
        consulta,
        conexion,
        parse_dates=["fecha"]
    )

print("Tipo del resultado:", type(datos_desde_sqlite))
display(datos_desde_sqlite.head())

### Actividad M8_8.23

1. Importa `mediciones.csv`.
2. Inspecciona forma, nombres y tipos.
3. Filtra un pozo y un periodo.
4. Ordena una variable.
5. Combina mediciones y pozos.
6. Calcula la cota piezométrica.
7. Resume por acuífero.
8. Crea una vista ancha.
9. Recupera la misma información desde SQLite.

---
# M8_8.24 · Visualización científica con Matplotlib

## 22. Figura y ejes

- La **figura** es el contenedor completo.
- Un **eje** es el área donde se dibuja un gráfico.

`plt.subplots()` crea ambos. Trabajar explícitamente con `figura` y `eje` facilita títulos, etiquetas, leyendas y exportación.

In [ ]:
figura, eje = plt.subplots(figsize=(7, 4))

eje.plot(
    serie_p01.index,
    serie_p01["profundidad_nivel_m"]
)

eje.set_title("Profundidad del nivel en P01")
eje.set_xlabel("Fecha")
eje.set_ylabel("Profundidad del nivel (m)")

plt.show()

## 23. Serie temporal

Una línea conecta observaciones ordenadas en el tiempo. Conviene mostrar puntos cuando la frecuencia de muestreo importa. Para profundidad medida desde superficie, invertir el eje puede representar visualmente mayor profundidad hacia abajo.

In [ ]:
figura, eje = plt.subplots(figsize=(10, 5))

for id_pozo, grupo in datos_completos.groupby("id_pozo"):
    grupo_ordenado = grupo.sort_values("fecha")

    eje.plot(
        grupo_ordenado["fecha"],
        grupo_ordenado["profundidad_nivel_m"],
        marker="o",
        label=id_pozo
    )

eje.invert_yaxis()
eje.set_title("Evolución de la profundidad del nivel")
eje.set_xlabel("Fecha")
eje.set_ylabel("Profundidad del nivel (m)")
eje.legend(title="Pozo", ncol=4)
figura.autofmt_xdate()

plt.tight_layout()
plt.show()

## 24. Diagrama de dispersión

Un scatter plot muestra pares de variables. Ayuda a observar dirección, forma, dispersión, grupos y valores alejados. No demuestra causalidad.

In [ ]:
figura, eje = plt.subplots(figsize=(7, 5))

for acuifero, grupo in datos_completos.groupby("acuifero"):
    eje.scatter(
        grupo["precipitacion_mm"],
        grupo["profundidad_nivel_m"],
        label=acuifero,
        alpha=0.7
    )

eje.set_title("Precipitación y profundidad del nivel")
eje.set_xlabel("Precipitación mensual (mm)")
eje.set_ylabel("Profundidad del nivel (m)")
eje.legend(title="Acuífero")

plt.show()

## 25. Histograma

El histograma agrupa valores en intervalos. Permite observar concentración, dispersión, asimetría y posibles grupos. La forma cambia con el número de intervalos, por lo que no debe interpretarse como una representación única.

In [ ]:
niveles_validos = datos_completos["profundidad_nivel_m"].dropna()

figura, ejes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

ejes[0].hist(niveles_validos, bins=6, edgecolor="black")
ejes[0].set_title("6 intervalos")
ejes[0].set_xlabel("Profundidad del nivel (m)")
ejes[0].set_ylabel("Número de observaciones")

ejes[1].hist(niveles_validos, bins=18, edgecolor="black")
ejes[1].set_title("18 intervalos")
ejes[1].set_xlabel("Profundidad del nivel (m)")

plt.tight_layout()
plt.show()

## 26. Comparación visual entre grupos

Mostrar observaciones individuales ayuda a ver tamaño de grupo, dispersión y solapamiento. Un valor resumen por sí solo puede ocultar estos aspectos.

In [ ]:
orden_acuiferos = sorted(datos_completos["acuifero"].dropna().unique())

figura, eje = plt.subplots(figsize=(8, 5))

for posicion, acuifero in enumerate(orden_acuiferos, start=1):
    valores = datos_completos.loc[
        datos_completos["acuifero"] == acuifero,
        "profundidad_nivel_m"
    ].dropna()

    desplazamiento = np.linspace(-0.08, 0.08, len(valores))

    eje.scatter(
        np.full(len(valores), posicion) + desplazamiento,
        valores,
        alpha=0.6
    )

eje.set_xticks(range(1, len(orden_acuiferos) + 1))
eje.set_xticklabels(orden_acuiferos)
eje.set_title("Profundidad del nivel por acuífero")
eje.set_xlabel("Acuífero")
eje.set_ylabel("Profundidad del nivel (m)")

plt.show()

## 27. Elementos mínimos de una figura científica

Una figura debe declarar:

- la pregunta o mensaje principal;
- nombres de variables;
- unidades;
- grupos y símbolos;
- escala temporal o espacial;
- tratamiento relevante de ausentes;
- fuente o procedimiento cuando se publica.

La interpretación debe distinguir lo observado de la explicación propuesta.

## 28. Decisiones gráficas que pueden inducir a error

- Recortar el eje puede exagerar diferencias.
- Unir puntos no ordenados puede sugerir una secuencia inexistente.
- Usar áreas o volúmenes dificulta comparar magnitudes.
- Ocultar puntos individuales puede ocultar tamaños de muestra.
- Mezclar unidades en un eje produce interpretaciones incorrectas.
- Una paleta decorativa no sustituye etiquetas y unidades.
- Una figura atractiva no corrige datos incorrectos.

In [ ]:
# El mismo conjunto puede parecer más o menos variable según los límites del eje.
datos_ejemplo = pd.Series([99.1, 99.4, 99.8, 100.2, 100.6])

figura, ejes = plt.subplots(1, 2, figsize=(11, 4))

for eje in ejes:
    eje.plot(datos_ejemplo.index, datos_ejemplo, marker="o")
    eje.set_xlabel("Observación")
    eje.set_ylabel("Valor")

ejes[0].set_ylim(0, 110)
ejes[0].set_title("Eje desde cero")

ejes[1].set_ylim(98.5, 101)
ejes[1].set_title("Eje recortado")

plt.tight_layout()
plt.show()

## 29. Exportar una figura

Los resultados regenerables se guardan en `data/output/figures`. Un nombre trazable identifica variable, lugar y periodo. `bbox_inches="tight"` evita recortes y `dpi` controla resolución.

In [ ]:
figura, eje = plt.subplots(figsize=(8, 4))

eje.plot(
    serie_p01.index,
    serie_p01["profundidad_nivel_m"],
    marker="o"
)

eje.invert_yaxis()
eje.set_title("Profundidad del nivel en P01")
eje.set_xlabel("Fecha")
eje.set_ylabel("Profundidad del nivel (m)")

ruta_figura = OUTPUT / "figures" / "P01_profundidad_nivel.png"

figura.savefig(
    ruta_figura,
    dpi=150,
    bbox_inches="tight"
)

plt.close(figura)
print("Figura guardada en:", ruta_figura)

## Actividad integradora

1. Selecciona un pozo.
2. Resume su serie por trimestre.
3. Calcula cota piezométrica.
4. Genera una serie temporal con unidades.
5. Genera una segunda figura que ayude a interpretar la variable.
6. Guarda ambas figuras en `data/output/figures`.
7. Escribe dos frases: patrón observado y limitación.

In [ ]:
id_seleccionado = "P02"

es_pozo_seleccionado = datos_completos["id_pozo"] == id_seleccionado

subconjunto = datos_completos.loc[
    es_pozo_seleccionado
].copy()

subconjunto = subconjunto.sort_values("fecha")

display(subconjunto.head())

## Síntesis

- NumPy representa arrays y permite cálculo vectorizado.
- pandas importa, organiza, selecciona, combina y resume DataFrames.
- Matplotlib convierte resultados en figuras interpretables.
- Los datos utilizados proceden de archivos y SQLite compartidos en el repositorio.
- La secuencia de trabajo es: localizar → importar → inspeccionar → seleccionar → transformar → comprobar → visualizar → guardar.